# Temporal Convolution Network: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import time

import numpy as np
import pandas as pd
import math
import matplotlib.dates as mdates
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.tcn_wrapper import TCNModel
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.data.dataset import TimeSeriesDataset
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "tcn"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"¡GPU Detectada! Usando: {torch.cuda.get_device_name(0)}")
else:
    print("ALERTA: No se detectó GPU")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)
ALERTA: No se detectó GPU


:::{admonition} Resumen
:class: info
- La optimización *Walk-Forward* con Optuna seleccionó la TCN con ventana de 14 días ($L = 14$), kernel de 3 ($K = 3$), 2 convoluciones por bloque, 5 niveles de dilatación y 64 canales (dropout $= 0.232$, tasa de aprendizaje $= 0.0072$, *weight decay* $= 6.0 \times 10^{-4}$).
- En la evaluación fuera de muestra con ventana expansiva y reentrenamiento iterativo alcanzó un QLIKE de $0.3675$ (frente a $0.4277$ en entrenamiento), superando a todos los modelos paramétricos evaluados y mostrando un desempeño ligeramente superior al de la GRU.
- Fuera de muestra absorbió la autocorrelación serial y los efectos ARCH residuales, presentando además un buen ajuste a una distribución *t*-Student ($\nu = 6.7266$).
- Persisten desviaciones de normalidad y dependencia no lineal detectada por la prueba BDS ($p = .028$), lo que motiva la exploración de esquemas híbridos con descomposición multiescala.
- Eleva su respuesta de forma más marcada que los modelos paramétricos ante el episodio de turbulencia de 2026, aunque conserva ligeros desfases temporales y no alcanza la magnitud puntual de los picos extremos.
:::

## Introducción

La Red Convolucional Temporal (*Temporal Convolutional Network*, TCN) es una arquitectura diseñada para modelar dependencias secuenciales mediante convoluciones causales y dilatadas. A diferencia de las redes recurrentes que procesan la secuencia paso a paso, la TCN aplica filtros en paralelo sobre la ventana temporal y expande su campo receptivo de forma exponencial con la profundidad mediante dilataciones, reduciendo el riesgo de desvanecimiento del gradiente.

En series financieras, esta arquitectura permite aprender relaciones no lineales entre retornos históricos y varianza futura directamente a partir de los datos, sin imponer una ecuación paramétrica rígida para la evolución de la varianza condicional.

El objetivo de esta sección es presentar, entrenar y evaluar un modelo TCN (`TCNModel`) para pronosticar la varianza de Rogers-Satchell del crudo Brent bajo el protocolo del *benchmark*. Se analiza el efecto de la longitud de ventana, el tamaño del kernel, el número de convoluciones por bloque, el número de niveles de dilatación, el ancho de canal y la regularización sobre el desempeño predictivo, manteniendo las mismas condiciones de información y evaluación que el resto de modelos del proyecto.

### Arquitectura

La arquitectura TCN se fundamenta en la combinación de tres componentes principales: convoluciones causales unidimensionales, convoluciones dilatadas y bloques residuales. A diferencia de las redes recurrentes que procesan la secuencia paso a paso, la TCN aplica filtros convolucionales en paralelo a lo largo de toda la ventana temporal, garantizando que la información fluya únicamente desde el pasado hacia el futuro.

Para una observación de entrada $\mathbf{x}_t$ y un filtro $\mathbf{w}_{\text{conv}}$ de tamaño $K$, la operación de convolución causal, que evita el sesgo de anticipación (*look-ahead bias*), se expresa como:

$$
\left(\mathbf{x} * \mathbf{w}_{\text{conv}}\right)[t] = \sum_{k=0}^{K-1} \mathbf{w}_{\text{conv}}[k] \cdot \mathbf{x}[t - k].
$$

Para capturar dependencias de largo plazo sin aumentar excesivamente el número de parámetros, se incorporaron **convoluciones dilatadas**. La dilatación introduce un espacio entre los elementos del filtro, permitiendo que el campo receptivo crezca exponencialmente con la profundidad. Para un nivel $l$, el factor de dilatación se define como $d = 2^l$. La operación resultante es:

$$
\left(\mathbf{x} *_{d} \mathbf{w}_{\text{conv}}\right)[t] = \sum_{k=0}^{K-1} \mathbf{w}_{\text{conv}}[k] \cdot \mathbf{x}[t - d \cdot k].
$$

Para estabilizar el entrenamiento en redes profundas, la TCN se compone de **bloques residuales**. Cada bloque contiene una o más convoluciones causales dilatadas (`num_convs`) seguidas de activaciones ReLU y capas de *Dropout*. La salida se suma a su entrada mediante una conexión residual (*skip connection*) y se aplica ReLU a la suma resultante:

$$
\mathbf{h}^{(l)} = \text{ReLU}\left( \mathcal{F}\left(\mathbf{h}^{(l-1)}\right) + \mathcal{R}\left(\mathbf{h}^{(l-1)}\right) \right),
$$

donde $\mathcal{F}(\cdot)$ representa la transformación convolucional del bloque, $\mathbf{h}^{(l)}$ el mapa de características en la capa $l$, y $\mathcal{R}(\cdot)$ la identidad o una convolución $1 \times 1$ cuando las dimensiones de entrada y salida difieren.

Una vez procesada la secuencia, se extrae el último vector temporal del mapa de características, $\mathbf{h}_t \in \mathbb{R}^{C}$ (donde $C$ es el número de canales de la última capa). Para generar un pronóstico escalar de la varianza condicional, este vector se proyecta a través de una capa densa con pesos $\mathbf{w}_{\text{fc}} \in \mathbb{R}^{C}$ y sesgo $b \in \mathbb{R}$:

$$
y_t = \mathbf{w}_{\text{fc}}^{\top}\mathbf{h}_t + b.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva, el resultado $y_t$ pasa por una función de activación *Softplus* junto con un término de estabilidad $\varepsilon = 10^{-8}$:

$$
\hat{\sigma}_t^2 = \text{Softplus}(y_t) + \varepsilon = \ln\left(1 + e^{y_t}\right) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ agrupa los pesos y sesgos tanto de las capas convolucionales como de la capa de salida.

Siguiendo el mismo esquema de preprocesamiento empleado en las arquitecturas recurrentes, los retornos logarítmicos diarios $r_t$ fueron estandarizados mediante una transformación Z-score para evitar el desvanecimiento de los gradientes. Para prevenir la fuga de información, el escalador se ajustó exclusivamente con el conjunto de entrenamiento de cada *fold*. Sea $\tilde{r}_t$ el retorno estandarizado, para una ventana de longitud $L$ (`look_back`), la secuencia de entrada se definió como $\mathbf{Z}_t \in \mathbb{R}^{L \times 1}$.

El problema se abordó mediante una estrategia de predicción de un paso hacia adelante (*one-step-ahead forecasting*). El entrenamiento consistió en encontrar los parámetros $\boldsymbol{\theta}$ que minimizaran la pérdida QLIKE sobre las observaciones disponibles, dado que esta métrica penaliza asimétricamente las subestimaciones de la volatilidad:

$$
\boldsymbol{\theta}^{*} = \arg\min_{\boldsymbol{\theta}} \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{QLIKE}}\left(RS_{i}, \widehat{RS}_{i}\right),
$$

donde $N$ es el número de secuencias de entrenamiento y $RS_i$ es la varianza de Rogers-Satchell utilizada como *target*.

Los parámetros se actualizaron mediante retropropagación del error estándar (*backpropagation*) y el optimizador *Adaptive Moment Estimation* (Adam), con *weight decay*, *Dropout* y parada temprana. La elección del tamaño del kernel ($K$), el número de niveles de dilatación, el número de convoluciones por bloque, el número de canales ocultos ($C$) y la longitud de las secuencias ($L$) influyó directamente sobre el campo receptivo y la capacidad de generalización del modelo. La implementación se realizó utilizando la librería `PyTorch` en su versión `2.13.0`, aprovechando su módulo de diferenciación automática.

## Entrenamiento

Para la selección de hiperparámetros del modelo TCN, se mantuvo el mismo protocolo experimental (Walk-Forward) y el mismo presupuesto computacional que en los modelos paramétricos y recurrentes. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la arquitectura de la red convolucional y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema Walk-Forward. Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento.

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), el tamaño del kernel (`kernel_size`), el número de convoluciones por bloque (`num_convs`), el número de canales ocultos (`channels`), la tasa de abandono (`dropout`) y la tasa de aprendizaje. Para evitar que la red procesara secuencias sin suficiente contexto temporal, el número de niveles de dilatación (`num_levels`) no se dejó completamente al azar: se calculó un límite inferior matemático utilizando la fórmula del campo receptivo, garantizando que la capa final pudiera "ver" la totalidad del `look_back` seleccionado. Para mitigar el sobreajuste, se incorporó también la penalización $\ell_2$ (`weight_decay`). 

El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def tcn_search_space(trial: optuna.Trial) -> dict:
    look_back = trial.suggest_categorical("look_back", [7, 14, 21, 28])
    kernel_size = trial.suggest_categorical("kernel_size", [2, 3])
    num_convs = trial.suggest_categorical("num_convs", [1, 2, 3])

    min_levels = math.ceil(
        math.log2(1 + (look_back - 1) / (num_convs * (kernel_size - 1)))
    )
    max_levels = min(min_levels + 2, 5)

    return {
        "look_back": look_back,
        "kernel_size": kernel_size,
        "num_convs": num_convs,
        "num_levels": trial.suggest_int("num_levels", min_levels, max_levels),
        "channels": trial.suggest_categorical("channels", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42
    }

exec_id = 6
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=TCNModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, warmup=current_warmup,
    embargo=1, n_trials=50,
    suggest_params_fn=tcn_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

El algoritmo de búsqueda se detuvo tras 19 ensayos completos de 50 posibles (38 %), con el mejor resultado alcanzado en el ensayo 3 y 0 ensayos podados. La ausencia de podas refleja que la red convolucional temporal procesa las secuencias en paralelo con convoluciones causales dilatadas y conexiones residuales, lo que favorece una convergencia estable en las primeras épocas sin requerir descartes prematuros durante la búsqueda.

La búsqueda completa en Optuna tomó 12 horas y 50 minutos (a `2572.47s/it`), mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó 6 minutos, con un promedio de 3.7800 segundos por partición. Este tiempo de ajuste evidencia el costo computacional de entrenar redes convolucionales profundas con Adam y retropropagación frente a modelos más ligeros, manteniendo la viabilidad operativa dentro del protocolo *Walk-Forward*.

In [2]:
resumen_data = []

exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="TCNModel_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
19,19,0,0.4370,359.1000,3.7800


El ranking de las cinco mejores configuraciones mostró un consenso en el uso de un tamaño de kernel $K = 3$, lo que sugiere que las convoluciones locales de tres días resultan informativas para capturar dinámicas de corto plazo en la volatilidad del Brent. En cuanto a la estructura temporal, las configuraciones más destacadas se concentraron en ventanas de 14 y 21 días y en profundidades de 4 y 5 niveles de dilatación. Al carecer de un estado oculto recurrente como las redes secuenciales tradicionales, la expansión exponencial del campo receptivo mediante convoluciones dilatadas permite integrar historia de mediano plazo y compensar la ausencia de recurrencia paso a paso.

In [3]:
exec_dir = RESULTS_PATH / "exec_6"

df_tcn = pd.read_csv(exec_dir / "TCNModel_results.csv")

df_tcn["lr"] = df_tcn["learning_rate"].apply(lambda x: f"{x:.4f}")
df_tcn["wd"] = df_tcn["weight_decay"].apply(lambda x: f"{x:.1e}")

cols_agrupar = ["look_back", "kernel_size", "num_convs", "num_levels", "channels", "dropout", "lr", "wd"]
df_agg = df_tcn.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()
df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "look_back": "Lookback",
    "kernel_size": "Kernel",
    "num_convs": "Convs",
    "num_levels": "Niveles",
    "channels": "Canales",
    "dropout": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay"
})

df_final = df_top5[["Lookback", "Kernel", "Convs", "Niveles", "Canales", "Dropout", "LR", "Weight Decay", "QLIKE"]]

tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Kernel,Convs,Niveles,Canales,Dropout,LR,Weight Decay,QLIKE
14,3,2,5,64,0.232,0.0072,6.0e-04,0.437 ± 0.111
21,3,1,5,128,0.235,0.0087,7.2e-04,0.454 ± 0.148
21,3,3,4,32,0.266,0.0071,4.0e-06,0.464 ± 0.154
21,3,1,5,128,0.231,0.0097,4.3e-04,0.464 ± 0.176
14,3,2,4,64,0.228,0.0046,3.4e-04,0.488 ± 0.221


Entre las dos primeras configuraciones, la diferencia de QLIKE es reducida frente a la dispersión observada entre *folds*, lo que hace que ambas sean cercanas en términos de validación. La elección de la primera configuración responde tanto a su menor pérdida media de validación como a un criterio de parsimonia al emplear 64 canales frente a 128, reduciendo la complejidad y la carga paramétrica del modelo sin sacrificar capacidad predictiva.

In [ ]:
best_model = TCNModel(
    look_back=14,
    kernel_size=3,
    num_convs=2,
    num_levels=5,
    channels=64,
    dropout=0.232,
    learning_rate=0.0072,
    weight_decay=6.0e-04,
    max_epochs=200,
    patience=20,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

El entrenamiento de la TCN sobre toda la muestra tuvo una duración de 5.0529 segundos. En la curva de aprendizaje, la pérdida de validación no diverge al alza ni abre una brecha creciente respecto a la de entrenamiento, por lo que no hay indicios de sobreajuste. Tras un pico inicial en entrenamiento (por encima del límite superior en la primera época) y oscilaciones moderadas, ambas trayectorias descienden y se estabilizan en valores bajos de QLIKE, con selección de mejores pesos en la época 20 y un comportamiento cercano entre ambas hasta la época 40.

In [8]:
history_csv_path = exec_dir / "tcn_training_history.csv"
history_df = pd.read_csv(history_csv_path)

if "epoch" in history_df.columns:
    history_df = history_df.drop(columns=["epoch"])

history_from_csv = {
    col: history_df[col].dropna().tolist()
    for col in history_df.columns
}

tt.plot_learning_curve(
    history=history_from_csv,
    title="Curva de Aprendizaje",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14, line_width=2.0,
    title_size=14, label_size=14, tick_size=12,
    legend_size=14, ylim=(0, 5), xlim=(0, 44),
    y_step=0.5, x_step=4, legend_bbox=(0.95, 0.95),
)

La pérdida de validación permanece por debajo de la de entrenamiento en buena parte de la serie debido a la misma asimetría temporal entre particiones ya señalada para el modelo GRU. Aunque la arquitectura (5 niveles de dilatación, 64 canales, kernel de 3 y 2 convoluciones por bloque) capta la dinámica general de la varianza condicional, la meseta posterior sugiere un margen limitado para refinar episodios extremos de volatilidad.

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no es posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) sobre la serie completa de entrenamiento: primero se transformó `y_train` con el `StandardScaler` ajustado durante `fit`, luego se construyó la matriz de rezagos con `TimeSeriesDataset` usando ventanas de 14 días (`look_back = 14`) y finalmente se evaluó en un solo lote en modo `eval()` sin gradientes ni regularización. Las varianzas estimadas se alinearon con los datos reales descartando las primeras 14 observaciones (`y_train[look_back:]`).

In [9]:
best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()

train_ds = TimeSeriesDataset(y_train_scaled, rs_train, look_back)
train_loader = DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)

with torch.no_grad():
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(best_model.device)
        preds_train_var = best_model.net(x_batch).cpu().numpy().flatten()

preds_train_var = preds_train_var / best_model.target_scale
rs_train_aligned = rs_train[look_back:]

df_train_preds = pd.DataFrame({
    "rs_true": rs_train_aligned,
    "y_pred": preds_train_var,
})

train_output_path = exec_dir / "tcn_train_predictions.csv"
df_train_preds.to_csv(train_output_path, index=False)

Para evaluar la estructura de los residuos sobre el conjunto de entrenamiento (In-Sample), a partir de las varianzas condicionales predichas $\left(\hat{\sigma}_t^2\right)$ y los retornos observados $\left(y_t\right)$, se calcularon los residuos estandarizados mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El diagnóstico sobre los residuos estandarizados de la TCN muestra ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .070$), así como ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .787$). En contraste, las pruebas ARCH-LM rechazan la hipótesis nula de homocedasticidad condicional en los tres horizontes evaluados ($p_{\text{adj}} < .001$), lo que indica que persiste agrupamiento residual de volatilidad dentro de la muestra de entrenamiento. Asimismo, las pruebas de Jarque-Bera y Lilliefors rechazan la normalidad de los residuos ($p < .001$), reflejando colas pesadas características de los retornos de volatilidad del crudo Brent.

In [15]:
train_output_path = exec_dir / "tcn_train_predictions.csv"

df_train_preds = pd.read_csv(train_output_path)
rs_train_aligned = df_train_preds["rs_true"].to_numpy()
preds_train_var = df_train_preds["y_pred"].to_numpy()

look_back = 14
y_train_aligned = y_train[look_back:]

std_residuals_train = y_train_aligned / np.sqrt(preds_train_var)
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),9.7940,.459,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),34.4333,.023,.070,Sin Autocorrelación
Ljung-Box (Lag 30),44.4671,.043,.130,Sin Autocorrelación
ARCH-LM (Lag 10),34.0167,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),55.2742,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),93.3536,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,1365.1707,< .001,--,Residuos No Normales
Lilliefors,0.0592,< .001,--,Residuos No Normales
BDS (Dim 2),-0.2704,.787,--,Sin Estructura No Lineal


Los residuos estandarizados de la TCN presentaron una estructura consistente con una distribución de colas pesadas, ajustándose a una distribución $t$ de Student. Esta correspondencia se observó en el histograma de densidad y en el gráfico Q-Q, donde los cuantiles muestrales siguieron la línea de referencia diagonal a lo largo de la mayor parte de la distribución. En los cuantiles extremos de ambas colas se aprecian desviaciones puntuales respecto a la recta teórica, lo que indica ligeras discrepancias en la captura de las observaciones más extremas. Pese a estas desviaciones en los extremos, las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .5420$; Cramér-von Mises: $p = .3699$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$.

In [11]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal (estadístico $= 247.16$, $p < .001$). Asimismo, los criterios de información mostraron una reducción de $\Delta\text{AIC} = 245.16$ y $\Delta\text{BIC} = 239.48$ a favor de la distribución $t$ de Student ($\nu = 4.1693$). Esto confirma que la distribución $t$ captura con mayor precisión las colas pesadas de los residuos estandarizados de la TCN en la muestra de entrenamiento en comparación con el ajuste Gaussiano.

In [12]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-3,479.89","6,963.78","6,975.13",-,-
t-Student,"-3,356.31","6,718.62","6,735.65",247.16,< .001


La inspección visual de las predicciones dentro de la muestra de entrenamiento muestra que la TCN responde a los cambios bruscos de volatilidad, aunque tiende a subestimar los episodios de alta volatilidad tanto en los shocks más extremos como en los repuntes de volatilidad moderada. En el recuadro de la crisis COVID-19 (`2020-01` a `2020-07`), la volatilidad predicha reacciona con impulsos marcados ante los movimientos de la serie realizada, pero no alcanza la magnitud de los máximos observados ni sostiene del todo los niveles elevados durante las fases de mayor perturbación. Un comportamiento similar se observa a lo largo de la muestra en episodios de volatilidad relativa moderada, donde la trayectoria estimada suele situarse por debajo de los picos de la volatilidad realizada.

In [19]:
look_back = 14

dates_aligned = df_train.index[look_back:]
rs_volatilidad_aligned = np.sqrt(rs_train[look_back:])
pred_volatilidad = np.sqrt(preds_train_var) 

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,      
    pred_variance=pred_volatilidad,
    title="Ajuste dentro la Muestra de Entrenamiento (TCN)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, 
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su ajuste sobre los datos observados con su capacidad predictiva en observaciones no vistas. Mientras que el conjunto de entrenamiento permite verificar la convergencia del algoritmo sobre la dinámica histórica, la evaluación fuera de muestra determina el desempeño del modelo ante nuevos regímenes de mercado.

Para diagnosticar posibles problemas de generalización, se aplicaron los criterios pre-registrados del protocolo experimental a partir de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{\text{abs}} = M_{\text{test}} - M_{\text{train}}
$$

Se fijó un umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ para las métricas evaluadas, de modo que discrepancias por encima de este margen señalan cambios relevantes entre el ajuste en muestra y el desempeño fuera de muestra.

En el modelado de volatilidad con pérdida QLIKE, esta lectura debe matizarse por la asimetría de la función de pérdida, que castiga con mayor severidad la subestimación de la varianza que su sobreestimación. Un modelo con sesgo alcista o defensivo puede registrar un QLIKE menor en prueba que en entrenamiento sin que ello implique una mejor capacidad estructural, sino una coincidencia favorable entre el nivel predicho y el régimen de volatilidad del periodo de prueba. Por ello, el análisis de $\Delta_{\text{abs}}$ se complementa con la inspección de las trayectorias predichas y el diagnóstico de residuos estandarizados.

Las predicciones fuera de muestra se generaron mediante el método `forecast` del modelo, aplicando una ventana expansiva con reentrenamiento periódico (`retrain_freq = 1`) sin fuga de información y devolviendo las varianzas condicionales en la escala original para su comparación directa con la volatilidad realizada.

In [ ]:
start_time = time.time()
preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test)
preds_time = time.time() - start_time

print(f"Tiempo de pronóstico: {preds_time / 60:.2f} minutos")

df_preds = pd.DataFrame({
    "date": df_test.index,
    "rs_true": rs_test,
    "y_pred": preds_test,
})

output_path = exec_dir / "tcn_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 109.38 minutos, lo que equivale a un promedio cercano a 6.89 segundos por predicción sobre las 953 observaciones del conjunto de prueba. Este tiempo refleja el costo computacional del reentrenamiento iterativo de la red convolucional temporal a lo largo del horizonte de prueba.

El análisis de los residuos estandarizados fuera de muestra presentó un comportamiento mixto. Las pruebas de Ljung-Box no rechazaron la hipótesis nula de ausencia de autocorrelación en ninguno de los rezagos evaluados ($p_{\text{adj}} \ge .172$), y las pruebas ARCH-LM tampoco detectaron efectos ARCH significativos en los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .799$), indicando que la TCN absorbió la dependencia lineal y el agrupamiento de volatilidad residual en datos no vistos. En cambio, la prueba BDS en dimensión 2 rechazó la hipótesis nula de independencia ($p = .028$), señalando la presencia de estructura no lineal residual en la muestra de prueba.

In [17]:
preds_path = exec_dir / "tcn_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].values

std_residuals_oos = y_test / np.sqrt(preds_test_var)
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),10.2942,.415,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),30.8520,.057,.172,Sin Autocorrelación
Ljung-Box (Lag 30),35.8389,.213,.640,Sin Autocorrelación
ARCH-LM (Lag 10),9.2132,.512,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),23.4693,.266,.799,Sin Efectos ARCH
ARCH-LM (Lag 30),30.5112,.440,1.000,Sin Efectos ARCH
Jarque-Bera,120.8101,< .001,--,Residuos No Normales
Lilliefors,0.0371,.005,--,Residuos No Normales
BDS (Dim 2),-2.1972,.028,--,Estructura No Lineal Significativa


El contraste entre la muestra de entrenamiento y la de prueba sugiere que el reentrenamiento iterativo durante la evaluación fuera de muestra favoreció la absorción de la heterocedasticidad condicional. Mientras que en entrenamiento persistieron efectos ARCH significativos en los rezagos 10, 20 y 30, en el conjunto de prueba las pruebas ARCH-LM no rechazaron la hipótesis nula de homocedasticidad condicional en ninguno de esos rezagos, acompañadas de ausencia de autocorrelación lineal en Ljung-Box. Esta mejora en la dinámica de varianza fuera de muestra es consistente con la actualización progresiva de los parámetros sobre la ventana expansiva, aunque la prueba BDS en dimensión 2 ($p = .028$) advierte sobre cierta dependencia no lineal remanente en prueba.

El comportamiento distribucional de los residuos estandarizados fuera de muestra mantuvo un buen ajuste a la distribución $t$ de Student, con un parámetro de grados de libertad estimado de $\nu = 6.7266$. Al ser mayor que el estimado en entrenamiento ($\nu = 4.1693$), este valor indica colas menos pesadas en la muestra de prueba frente a la dinámica histórica de entrenamiento. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .7581$; Cramér-von Mises: $p = .7530$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$. En el gráfico Q-Q, los cuantiles muestrales se alinean con la diagonal teórica en la mayor parte del rango, con ligeras desviaciones en los extremos.

In [21]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La TCN excedió el umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ en QLIKE ($\Delta_{\text{abs}} = -0.0601$), con menor pérdida en prueba que en entrenamiento. Este resultado responde a la asimetría de QLIKE y al régimen de volatilidad del periodo de prueba, y también puede asociarse a un ligero subajuste en el ajuste histórico frente a la dinámica de los mercados evaluados, mientras que en prueba el pronóstico *one-step* con ventana expansiva se adapta de forma progresiva al avance de la serie. Frente al mejor QLIKE de validación ($0.4370$), el desempeño en prueba ($0.3675$) mejora apoyado en ese reentrenamiento iterativo.

En el resto de las métricas, las diferencias entre entrenamiento y prueba se mantuvieron dentro del margen de tolerancia y en magnitudes bajas, con descensos en prueba en las métricas ajustadas por heterocedasticidad (HMSE y HMAE). En términos generales, la TCN superó a todos los modelos paramétricos evaluados y mostró un rendimiento ligeramente superior al del modelo GRU en el conjunto de prueba. El análisis de significancia estadística de las diferencias en poder predictivo se abordará en otra libreta.

In [22]:
look_back = 14
rs_train_aligned = rs_train[look_back:] 

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba", 
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4277,0.3675,-0.0601
CRPS,0.0129,0.0123,-0.0006
Pinball (τ=0.05),0.0026,0.0027,+0.0001
Pinball (τ=0.95),0.0015,0.0015,+0.0000
MSE,0.0000,0.0000,+0.0000
RMSE,0.0020,0.0022,+0.0002
MAE,0.0004,0.0005,+0.0000
HMSE,2.3805,1.7030,-0.6775
HMAE,0.7722,0.6482,-0.1240


La inspección visual del ajuste fuera de la muestra para la TCN muestra un seguimiento cercano de la dinámica general de la volatilidad realizada desde finales de 2022 hasta mediados de 2026. En el recuadro de acercamiento al pico de volatilidad de 2026 (`2026-02` a `2026-07`), el modelo eleva su predicción de forma más marcada que los modelos paramétricos ante el episodio de turbulencia, aunque sigue sin alcanzar el máximo puntual de la serie realizada y presenta ligeros desfases temporales en los repuntes posteriores. En regímenes de volatilidad normal o moderada, la predicción alterna entre ligera sobreestimación y subestimación puntual, pero en promedio sigue la tendencia esperada de la serie realizada.

In [24]:
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_vol = np.sqrt(df_preds_test["rs_true"])
pred_test_vol = np.sqrt(df_preds_test["y_pred"])

mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (df_preds_test["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,      
    pred_variance=pred_test_vol,
    title="Ajuste fuera de la Muestra de Entrenamiento (TCN)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Conclusiones

La evaluación del modelo TCN con ventana de 14 días, kernel de 3, 2 convoluciones por bloque, 5 niveles de dilatación y 64 canales registró una pérdida QLIKE fuera de muestra de $0.3675$, superando a todos los modelos paramétricos evaluados y mostrando un rendimiento ligeramente superior al del modelo GRU. Estos resultados confirman que las convoluciones causales dilatadas combinadas con bloques residuales y el reentrenamiento iterativo sobre ventana expansiva permiten capturar relaciones temporales no lineales de corto y mediano plazo en la volatilidad del crudo Brent.

En el conjunto de prueba, el modelo redujo el QLIKE respecto al ajuste histórico ($0.3675$ frente a $0.4277$) y disminuyó las métricas ajustadas por heterocedasticidad (HMSE y HMAE), elevando su predicción de forma más marcada que las especificaciones paramétricas ante el episodio de turbulencia de 2026. No obstante, la inspección visual evidencia que persiste cierto desfase temporal ante saltos abruptos de régimen y que el modelo no alcanza la magnitud puntual de los picos extremos, alternando ligeras subestimaciones y sobreestimaciones en periodos de volatilidad moderada, lo que indica que aún existe margen de mejora en la reactividad inmediata.

El diagnóstico de residuos estandarizados fuera de muestra confirmó que el modelo absorbió la autocorrelación serial y los efectos ARCH residuales, presentando además un buen ajuste a una distribución *t*-Student ($\nu = 6.7266$). A diferencia de los modelos paramétricos, donde la dependencia no lineal remanente era recurrente, las arquitecturas profundas logran mitigar gran parte de esa estructura compleja, aunque la prueba BDS ($p = .028$) señala que persiste cierta no linealidad residual en el conjunto de prueba, motivando la exploración de esquemas híbridos que combinen descomposición multiescala con redes profundas.